# 15 · Dispersión dinámica: ¿puede una red reorganizar sus conexiones mientras aprende?

**Resultado: negativo.** RNRA-v0 (reasignar conexiones según utilidad, redundancia y demanda por región, con neuronas de reserva y una prueba «con y sin cambio») **no superó a RigL ni al azar** con 10 semillas, y se quitó del código (commit `fd6e8f3`; informe: `laboratorio/banco/informes/15-reorganizacion-autonoma.md`). Lo que queda en este cuaderno es el **banco** para seguir probando ideas de redes dispersas contra comparadores fuertes, todos con el mismo presupuesto de conexiones (las de un MLP denso de 16 neuronas por capa):

`mlp_pequeno` · `mlp_grande` · `poda_magnitud` (parte del grande y poda de a poco) · `crecimiento_programado` · `set` (crea al azar; también es el control aleatorio) · `rigl` (crea donde el gradiente es mayor).

**Límites:** los pesos se guardan con el tamaño máximo y se enmascaran, así que apagar conexiones no ahorra tiempo ni memoria reales; los FLOPs son lógicos (conexiones activas). Sirve CPU; no necesita GPU.

## 1. Preparar (≈ 2 min)

In [ ]:
import os, subprocess
from google.colab import userdata

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt

In [ ]:
import numpy as np
from dataclasses import replace
from laboratorio.banco.tareas import cargar_tarea
from laboratorio.rnra import experimentos as ex
from laboratorio.rnra.controlador import METODOS, Config

SEMILLAS = range(10)       # baja a 3 para probar
CFG = Config()             # 800 pasos
print(METODOS)

## 2. E1 y E2 · xor, círculos, lunas y caótica 20D (≈ 4 min)

In [ ]:
for nombre in ex.ESTUDIOS["E1"] + ex.ESTUDIOS["E2"]:
    res = ex.correr(cargar_tarea(nombre), METODOS, SEMILLAS, CFG)
    print(f"\n### {nombre}\n"); print(ex.tabla(res)); print(); print(ex.tabla_comparacion(res, "rigl"))

## 3. E3 y E4 · cambios de distribución y aprendizaje continuo (≈ 6 min)

In [ ]:
for sec in (ex.lunas_rotadas(), ex.digitos_permutados(), ex.digitos_partidos()):
    res = ex.correr_continuo(sec, METODOS, SEMILLAS, CFG)
    print(f"\n### {sec.nombre}")
    for m, rs in res.items():
        print(f"{m:24s} final {np.mean([r.metrica_prueba for r in rs])*100:5.1f} %   olvido {np.mean([r.perdida_prueba for r in rs])*100:5.1f} %")
    print(ex.tabla_comparacion(res, "rigl"))

## 4. E5 · Fashion-MNIST (≈ 15 min, baja datos de Hugging Face)

Para 784 entradas el presupuesto y las neuronas se escalan (`ancho_pequeno=64`).

In [ ]:
t5 = cargar_tarea("fashion_mnist")
cfg5 = replace(CFG, pasos=1500, ancho_pequeno=64, activas_inicial=96, ancho_max=192, ancho_grande=192, lote=256)
res5 = ex.correr(t5, METODOS, range(5), cfg5)
print(ex.tabla(res5)); print(); print(ex.tabla_comparacion(res5, "rigl"))

## 5. Guardar el informe en Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/Notbook/resultados_rnra", exist_ok=True)
!python -m laboratorio.rnra --todo --semillas 10 --salida /content/drive/MyDrive/Notbook/resultados_rnra/15-dispersion-dinamica.md > /dev/null
print("Guardado en Drive: Notbook/resultados_rnra/")